# Multi-head Latent Attention (MLA) — a toy-scale build

A minimal implementation of **Multi-head Latent Attention**, the attention
variant introduced in DeepSeek-V2 (DeepSeek-AI, 2024) and carried forward
into DeepSeek-V3 — designed to shrink the KV cache dramatically without
changing what the attention mechanism computes.

Companion write-up: `README.md` in this folder. (This is also the mechanism
Kimi K3's Gated MLA layer is named after and simplifies away — see `../kda`
for that architecture's take, which drops the latent compression entirely
since it's a serving optimization, not a semantics change. This notebook
builds the compression itself, since it's the interesting part.)

## 0. Setup

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

## 1. The problem MLA solves

During autoregressive generation, a transformer caches every past token's
keys and values so it doesn't have to recompute them at every new step —
this is the "KV cache." The catch: the KV cache's size grows with
`n_heads * d_head` per token, and for models with many heads and long
context, that cache can become the dominant memory cost of serving the
model — often bigger than the model's own weights.

**MLA's idea:** instead of caching a full-size key and value per head,
compress all heads' keys and values down into one small shared **latent
vector** per token, and cache *that* instead. When you actually need the
per-head keys and values for attention, up-project from the latent on the
fly.

```
c_t = W_down x_t              # one small latent vector per token -- THIS is what gets cached
k_t^(h) = W_up_k^(h) c_t        # reconstructed per-head key, computed on demand
v_t^(h) = W_up_v^(h) c_t        # reconstructed per-head value, computed on demand
```

If the latent is much smaller than `n_heads * d_head` combined, the cache
shrinks by roughly that ratio — while the attention computation itself,
once the up-projection happens, is completely ordinary multi-head attention.
**MLA doesn't change what attention computes — it changes what gets stored
between tokens.**

## 2. The one wrinkle: positional encoding

Most modern transformers use **RoPE** (rotary position embeddings) baked
directly into the keys and queries. But RoPE only works cleanly if the key
you're rotating is stable and specific to a token — and MLA's whole point is
that the *cached* thing (the latent `c_t`) is shared across all heads and
gets a fresh up-projection every time you use it, which doesn't play well
with rotating it once and caching the rotated version.

MLA's fix: split off a **small, separate "rope" head** that carries
positional information directly (computed from the raw token, RoPE applied,
*not* compressed through the latent), and concatenate it onto each head's
content key before computing attention scores. So each head's effective key
is `[reconstructed_content_key ; shared_rope_key]` — content comes from the
compressed latent, position comes from an uncompressed side channel.

> **Simplification used here:** in a production implementation, the
> up-projection matrices are often algebraically folded into the query/output
> projections to avoid materializing full-size keys and values at all
> (an "absorption" trick purely for serving efficiency). This notebook
> materializes them explicitly with `k_up`/`v_up`, since it computes the
> exact same attention output and is far easier to follow.

In [ ]:
def rotate_half(x):
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat([-x2, x1], dim=-1)

def apply_rope(x, cos, sin):
    return x * cos + rotate_half(x) * sin

class MLA(nn.Module):
    def __init__(self, d_model=64, n_heads=2, d_head=32, d_latent=16, d_rope=8):
        super().__init__()
        self.h, self.dh, self.d_rope = n_heads, d_head, d_rope
        inner = n_heads * d_head

        self.q_proj = nn.Linear(d_model, inner, bias=False)
        self.q_rope_proj = nn.Linear(d_model, n_heads * d_rope, bias=False)

        self.kv_down = nn.Linear(d_model, d_latent, bias=False)      # the ONLY thing that gets cached
        self.kv_latent_norm = nn.LayerNorm(d_latent)
        self.k_up = nn.Linear(d_latent, inner, bias=False)
        self.v_up = nn.Linear(d_latent, inner, bias=False)
        self.k_rope_proj = nn.Linear(d_model, d_rope, bias=False)     # shared rope key, uncompressed

        self.out_proj = nn.Linear(inner, d_model, bias=False)
        self.scale = (d_head + d_rope) ** -0.5

    def forward(self, x):
        B, T, D = x.shape
        H, Dh, Dr = self.h, self.dh, self.d_rope
        device = x.device

        pos = torch.arange(T, device=device).float()
        inv_freq = 1.0 / (10000 ** (torch.arange(0, Dr, 2, device=device).float() / Dr))
        freqs = torch.einsum('t,d->td', pos, inv_freq)
        emb = torch.cat([freqs, freqs], dim=-1)
        cos, sin = emb.cos()[None, :, None, :], emb.sin()[None, :, None, :]

        q_c = self.q_proj(x).view(B, T, H, Dh)
        q_r = apply_rope(self.q_rope_proj(x).view(B, T, H, Dr), cos, sin)

        c_kv = self.kv_latent_norm(self.kv_down(x))         # B,T,d_latent -- the compressed cache
        k_c = self.k_up(c_kv).view(B, T, H, Dh)               # reconstructed per-head content key
        v_c = self.v_up(c_kv).view(B, T, H, Dh)
        k_r = self.k_rope_proj(x)                              # B,T,Dr -- shared rope key
        k_r = apply_rope(k_r.unsqueeze(2), cos, sin).squeeze(2).unsqueeze(2).expand(B, T, H, Dr)

        q = torch.cat([q_c, q_r], dim=-1).transpose(1, 2)     # B,H,T,Dh+Dr
        k = torch.cat([k_c, k_r], dim=-1).transpose(1, 2)
        v = v_c.transpose(1, 2)

        attn = torch.einsum('bhtd,bhsd->bhts', q, k) * self.scale
        mask = torch.triu(torch.ones(T, T, device=device, dtype=torch.bool), diagonal=1)
        attn = attn.masked_fill(mask, float('-inf')).softmax(-1)
        o = torch.einsum('bhts,bhsd->bhtd', attn, v).transpose(1, 2).reshape(B, T, H * Dh)
        return self.out_proj(o)

## 3. Assembling a tiny language model

In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))
    def forward(self, x):
        norm = x.pow(2).mean(-1, keepdim=True)
        return x * torch.rsqrt(norm + self.eps) * self.weight

class SwiGLU(nn.Module):
    def __init__(self, d, hidden_mult=2):
        super().__init__()
        h = d * hidden_mult
        self.Wg = nn.Linear(d, h, bias=False)
        self.Wu = nn.Linear(d, h, bias=False)
        self.Wd = nn.Linear(h, d, bias=False)
    def forward(self, x):
        return self.Wd(F.silu(self.Wg(x)) * self.Wu(x))

class TinyLM(nn.Module):
    def __init__(self, vocab_size, d_model=64, n_layers=2):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model)
        self.blocks = nn.ModuleList([MLA(d_model) for _ in range(n_layers)])
        self.mlps = nn.ModuleList([SwiGLU(d_model) for _ in range(n_layers)])
        self.norms1 = nn.ModuleList([RMSNorm(d_model) for _ in range(n_layers)])
        self.norms2 = nn.ModuleList([RMSNorm(d_model) for _ in range(n_layers)])
        self.final_norm = RMSNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)

    def forward(self, idx):
        x = self.embed(idx)
        for blk, mlp, n1, n2 in zip(self.blocks, self.mlps, self.norms1, self.norms2):
            x = x + blk(n1(x))
            x = x + mlp(n2(x))
        return self.lm_head(self.final_norm(x))

## Proving it actually works

Everything above is only worth something if gradients actually flow correctly
through MLA once it's wired into a real model. So the rest of this
notebook:

1. wraps MLA into a tiny 2-layer causal language model,
2. builds a **tiny synthetic dataset** (a repeating `"0123456789ABCDEF"`
   string — enough to check the model can learn *any* sequential structure
   at all, no real corpus needed),
3. runs **one forward + backward pass** as a sanity check (right output
   shape, no `NaN` gradients),
4. **trains for a few hundred steps**, and
5. **generates** from the trained model — if training worked, the output
   should show visible periodicity.

This is deliberately not a "real" training run. It exists purely to catch
architecture bugs, which is the whole point of a toy-scale build.

In [ ]:
# --- synthetic dataset ---
pattern = "0123456789ABCDEF"      # synthetic, no copyright concerns
text = pattern * 200
chars = sorted(set(text))
stoi = {c: i for i, c in enumerate(chars)}
itos = {i: c for c, i in stoi.items()}
data = torch.tensor([stoi[c] for c in text], dtype=torch.long)
vocab_size = len(chars)
max_seq_len = 32

model = TinyLM(vocab_size).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f"Model built. Trainable parameters: {n_params:,}")

In [ ]:
# --- sanity check: one forward + backward pass before training ---
xb0 = data[:max_seq_len].unsqueeze(0).to(device)
yb0 = data[1:max_seq_len + 1].unsqueeze(0).to(device)
out0 = model(xb0)
logits0 = out0[0] if isinstance(out0, tuple) else out0
print(f"Sanity check -- logits shape: {tuple(logits0.shape)} (expect [1, {max_seq_len}, {vocab_size}])")
loss0 = F.cross_entropy(logits0.reshape(-1, vocab_size), yb0.reshape(-1))
if isinstance(out0, tuple):
    loss0 = loss0 + out0[1]
loss0.backward()
n_nan_grads = sum(torch.isnan(p.grad).any().item() for p in model.parameters() if p.grad is not None)
print(f"Sanity check -- initial loss: {loss0.item():.4f}, NaN grads: {n_nan_grads}")
model.zero_grad()

In [ ]:
# --- training loop ---
def get_batch(data, block_size, batch_size, device):
    ix = torch.randint(0, len(data) - block_size - 1, (batch_size,))
    x = torch.stack([data[i:i + block_size] for i in ix])
    y = torch.stack([data[i + 1:i + block_size + 1] for i in ix])
    return x.to(device), y.to(device)

opt = torch.optim.AdamW(model.parameters(), lr=3e-3)
n_steps, batch_size = 300, 16
print("Training on synthetic periodic sequence (verifies grads flow end-to-end)...")
for step in range(n_steps):
    xb, yb = get_batch(data, max_seq_len, batch_size, device)
    out = model(xb)
    logits = out[0] if isinstance(out, tuple) else out
    loss = F.cross_entropy(logits.reshape(-1, vocab_size), yb.reshape(-1))
    if isinstance(out, tuple):
        loss = loss + out[1]
    opt.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    opt.step()
    if step % 50 == 0 or step == n_steps - 1:
        print(f"  step {step:4d} | loss {loss.item():.4f}")

In [ ]:
# --- generation ---
@torch.no_grad()
def generate(model, start_idx, n_new):
    model.eval()
    idx = start_idx.clone()
    for _ in range(n_new):
        out = model(idx)
        logits = out[0] if isinstance(out, tuple) else out
        probs = F.softmax(logits[:, -1, :], dim=-1)
        next_id = torch.multinomial(probs, num_samples=1)
        idx = torch.cat([idx, next_id], dim=1)
    model.train()
    return idx

start = data[:8].unsqueeze(0).to(device)
gen = generate(model, start, 48)[0].tolist()
print("Generated (should show visible periodicity if training worked):")
print(''.join(itos[i] for i in gen))

## Where to go from here

- **Measure the actual cache savings.** Print `d_latent + d_rope` vs.
  `n_heads * d_head * 2` (the size of a standard KV cache per token) at a
  few different configs, and see how the compression ratio scales as you
  add more heads.
- **Try the "absorption" trick** — algebraically fold `k_up` into the
  attention score computation so you never materialize the full-size key at
  all. It produces identical outputs and is the actual serving-time
  optimization real MLA implementations use.
- **Compare against NSA** (`../nsa`) — a different DeepSeek idea for making
  attention cheaper, this time by making it *sparse* rather than
  compressing what gets cached.

Reference: DeepSeek-AI, *"DeepSeek-V2: A Strong, Economical, and Efficient
Mixture-of-Experts Language Model,"* 2024.